# Сборка рабочего датасета

Этот ноутбук собирает итоговый датасет: берёт результаты LLM-классификации, добавляет цену победителя и НМЦК, флаг пакетности, NLP-признаки из документов, приводит цены к 2025 году и объединяет публикации в объекты строительства.

In [214]:
import pandas as pd
import numpy as np
import re
from pathlib import Path
from difflib import SequenceMatcher

ROOT = Path(".")

# веса для дефлятора СМР (труд / материалы / машины)
W_LABOR = 0.25
W_MATERIALS = 0.65
W_MACHINES = 0.10
BASE_QUARTER = "2025Q4"

Загрузка данных:

In [215]:
cls = pd.read_csv(ROOT / "data_processed" / "marker_results_classified.csv")
main = pd.read_csv(ROOT / "data_processed" / "marker_results_main_clean.csv")
parts = pd.read_csv(ROOT / "data_processed" / "marker_results_participants_clean.csv")
parsed = pd.read_csv(ROOT / "data_processed" / "marker_docs_parsed_auto.csv")

print(f"classified: {len(cls)} строк")
print(f"main: {len(main)} строк (публикации)")
print(f"participants: {len(parts)} строк (участники)")
print(f"parsed: {len(parsed)} строк")

classified: 157 строк
main: 157 строк (публикации)
participants: 183 строк (участники)
parsed: 156 строк


In [216]:
included_ids = set(cls[cls["include_base"] == True]["registry_number"].astype(str))
print(f"Включено по LLM-классификации: {len(included_ids)} публикаций")

df = main[main["registry_number"].astype(str).isin(included_ids)].copy()
print(f"После фильтрации main: {len(df)} строк")

Включено по LLM-классификации: 111 публикаций
После фильтрации main: 111 строк


Добавляю цену победителя и НМЦК

In [217]:
winners = parts[
    parts["Победитель"].astype(str).str.lower().str.contains("победитель", na=False)
][["registry_number", "supplier_price_rub", "discount_pct"]].copy()
winners = winners.drop_duplicates(subset="registry_number", keep="first")
winners.columns = ["registry_number", "contract_price_nom", "discount_pct"]

print(f"Публикаций с найденным победителем: {len(winners)}")

Публикаций с найденным победителем: 74


In [218]:
df.head(3)

,Уровень,Заказчик,ИНН заказчика,customer_price_rub,registry_number,purchase_code,Сфера деятельности,publication_name,Регион поставки,Город поставки,...,Источник финансирования,Нацрежим,Ссылка на источник,Цвет,Комментарий,publication_url,publication_name_full,source_url,source_text,_source_file
0,1,"МБОУ ""СЕВЕРНАЯ СОШ №2""",5.645002e+09,5.134048e+05,6448182,NaN,[ОКПД2 41.20] Здания и работы по возведению зд...,Капитальное строительство центра образования е...,Оренбургская область,Северный район,...,Неизвестно,Нет,РТС-ТЕНДЕР 44-ФЗ,NaN,NaN,https://analytics.marker-zakupki.ru/Card/Lot/1...,Капитальное строительство центра образования е...,https://market.rts-tender.ru/zapros/6448182,РТС-ТЕНДЕР 44-ФЗ,результаты.xlsx
2,1,АДМИНИСТРАЦИЯ МР ХАЙБУЛЛИНСКИЙ РАЙОН РБ,2.480052e+08,3.332589e+08,101500000322000125,22-30248005212024801001-0031-001-4120-414,[ОКПД2 41.20] Здания и работы по возведению зд...,Школа на 550 мест с интернатом на 140 мест в с...,Республика Башкортостан,Республика Башкортостан,...,Бюджет Республики Башкортостан; Бюджет муницип...,Нет,Госзакупки 44ФЗ/94ФЗ,NaN,NaN,https://analytics.marker-zakupki.ru/Card/Lot/1...,Школа на 550 мест с интернатом на 140 мест в с...,https://zakupki.gov.ru/epz/order/notice/ok20/v...,Госзакупки 44ФЗ/94ФЗ,результаты.xlsx
3,1,ГКУ УКС РБ,2.781765e+08,6.772817e+08,101500000322000183,22-20278176470027601001-0348-001-4120-414,[ОКПД2 41.20] Здания и работы по возведению зд...,"Выполнение строительно-монтажных, пусконаладоч...",Республика Башкортостан,Абзелиловский район,...,Неизвестно,Нет,Госзакупки 44ФЗ/94ФЗ,NaN,NaN,https://analytics.marker-zakupki.ru/Card/Lot/1...,"Выполнение строительно-монтажных, пусконаладоч...",https://zakupki.gov.ru/epz/order/notice/ok20/v...,Госзакупки 44ФЗ/94ФЗ,результаты.xlsx


In [219]:
df.columns

Index(['Уровень', 'Заказчик', 'ИНН заказчика', 'customer_price_rub',
       'registry_number', 'purchase_code', 'Сфера деятельности',
       'publication_name', 'Регион поставки', 'Город поставки',
       'Дата публикации',
       'Дата окончания приема заявок / Дата планового окончания исполнения контракта / Плановая дата публикации лота по ППГ',
       'Дата начала подачи заявок/Дата начала исполнения контракта / Дата публикации ППГ',
       'Дата окончания проведения торгов', 'Поставщик', 'ИНН поставщика',
       'Победитель', 'Статус допуска', 'supplier_price_rub', 'discount_pct',
       'publication_form', 'Тип торгов', 'Торговая площадка',
       'Электронные торги', 'Обеспечение заявки (руб.)',
       'Обеспечение заявки, %', 'Обеспечение контракта (руб.)',
       'Обеспечение контракта, %', 'Банковское \ казначейское сопровождение',
       'Источник финансирования', 'Нацрежим', 'Ссылка на источник', 'Цвет',
       'Комментарий', 'publication_url', 'publication_name_full', 'sour

In [220]:
df = df.drop(columns=["discount_pct"])

df = df.merge(winners, on="registry_number", how="left")
df["has_contract_price"] = df["contract_price_nom"].notna()
df.rename(columns={"customer_price_rub": "nmck_nom"}, inplace=True)

# Основная денежная переменная - цена контракта при наличии, иначе НМЦК
df["price_nom"] = df["contract_price_nom"].fillna(df["nmck_nom"])
df["price_source"] = np.where(df["has_contract_price"], "contract", "nmck")

print(f"Из {len(df)} публикаций:")
print(f"  цена контракта: {df['has_contract_price'].sum()}")
print(f"  только НМЦК: {(~df['has_contract_price']).sum()}")

Из 111 публикаций:
  цена контракта: 56
  только НМЦК: 55


Статистика price_nom

In [221]:
print((df["price_nom"] / 1e6).describe().round(1))

count     111.0
mean      793.0
std       639.9
min         0.5
25%       313.2
50%       618.5
75%      1262.9
max      3536.9
Name: price_nom, dtype: float64


### Проверяю, нет ли в закупке мебели/оборудования (пакетные закупки)

In [222]:
PACKAGE_KEYWORDS = [
    "мебель", "инвентарь", "компьютер", "ноутбук", "оргтехника",
    "учебное оборудование", "мультимедиа", "проектор", "интерактивная доска",
    "питание", "охрана", "клининг", "уборка",
]

prices_df = pd.read_excel(ROOT / "data строительство" / "цены.xlsx")
prices_df["name_low"] = prices_df["Наименование товара"].astype(str).str.lower()

pkg_flags = (
    prices_df.groupby("Реестровый номер публикации")["name_low"]
    .agg(lambda x: any(kw in " ".join(x.dropna()) for kw in PACKAGE_KEYWORDS))
    .reset_index()
)
pkg_flags.columns = ["registry_number", "package_purchase"]
pkg_flags["registry_number"] = pkg_flags["registry_number"].astype(str)

df["registry_number"] = df["registry_number"].astype(str)
df = df.merge(pkg_flags, on="registry_number", how="left")
df["package_purchase"] = df["package_purchase"].fillna(False)

print(f"Флаг пакетности: {df['package_purchase'].sum()} публикаций из {len(df)}")

Флаг пакетности: 0 публикаций из 111


Добавляю число мест, площадь и адрес из парсинга документов

In [223]:
parsed_cols = parsed[[
    "registry_number", "places", "places_source_file", "places_snippet",
    "area_m2", "address_text", "confidence"
]].copy()
parsed_cols["registry_number"] = parsed_cols["registry_number"].astype(str)

drop_cols = [c for c in ["places", "places_source_file", "places_snippet",
                          "area_m2", "address_text", "confidence"] if c in df.columns]
df = df.drop(columns=drop_cols)
df = df.merge(parsed_cols, on="registry_number", how="left")

print(f"После мержа с парсингом: {len(df)} публикаций")
print(f"  places извлечены: {df['places'].notna().sum()} ({df['places'].notna().mean():.1%})")
print(f"  area_m2 извлечены: {df['area_m2'].notna().sum()} ({df['area_m2'].notna().mean():.1%})")
print(f"  address извлечен: {df['address_text'].notna().sum()} ({df['address_text'].notna().mean():.1%})")

После мержа с парсингом: 111 публикаций
  places извлечены: 109 (98.2%)
  area_m2 извлечены: 86 (77.5%)
  address извлечен: 80 (72.1%)


In [224]:
print(f"Публикации без places: {df['places'].isna().sum()}")
print(df[df['places'].isna()][["registry_number", "publication_name"]].to_string())

Публикации без places: 2
      registry_number                                                                                                                                                                                                         publication_name
0             6448182  Капитальное строительство центра образования естественно-научной и технологической направленностей "Точка роста" МБОУ "Северная СОШ №2" по адресу: Оренбургская область, Северный район, с. Северное, ул. Осенняя, д. 2
6  101500000322000403           Выполнение строительно-монтажных, пусконаладочных работ, поставка оборудования, неразрывно связанного с производством работ, по объекту "Строительство школы в МР-3 Восточного жилого района ГО г. Салават РБ"


Перевожу дату в квартал (нужно для дефлятора)

Распределение по кварталам:

In [225]:
df["date_pub"] = pd.to_datetime(df["Дата публикации"], unit="D", origin="1899-12-30")
df["year"] = df["date_pub"].dt.year
df["quarter"] = df["date_pub"].dt.quarter
df["quarter_code"] = df["year"].astype(str) + "Q" + df["quarter"].astype(str)

print(df.groupby("quarter_code").size().sort_index())

quarter_code
2022Q1     4
2022Q2    11
2022Q3     5
2022Q4     7
2023Q1     9
2023Q2     1
2023Q3     5
2023Q4     5
2024Q1     3
2024Q2    13
2024Q3     3
2024Q4     8
2025Q1    15
2025Q2    11
2025Q3     4
2025Q4     7
dtype: int64


Дефлятирую цены к уровню 2025Q4

Умножаю каждый элемент на его вес и суммирую — получаю итоговый дефлятор

In [226]:
idx_long = pd.read_csv(ROOT / "data_processed" / "minstroy_school_indices_2022Q1_2025Q4_long_filled.csv")

weights = {"labor": W_LABOR, "materials": W_MATERIALS, "machines": W_MACHINES}
idx_long["weighted"] = idx_long["index_value"] * idx_long["element"].map(weights)
idx_wide = idx_long.groupby(["quarter", "region"])["weighted"].sum().reset_index()
idx_wide.columns = ["quarter", "region", "I_smr"]


Базовый индекс - 2025Q4 - для каждого региона

In [227]:
base = idx_wide[idx_wide["quarter"] == BASE_QUARTER][["region", "I_smr"]].copy()
base.columns = ["region_std", "I_smr_base"]

df["region_std"] = df["Регион поставки"]

for col in ["I_smr_curr", "I_smr_base", "price_base", "nmck_base"]:
    if col in df.columns:
        df = df.drop(columns=[col])

idx_curr = idx_wide.rename(columns={"quarter": "quarter_code", "region": "region_std", "I_smr": "I_smr_curr"})
df = df.merge(idx_curr, on=["quarter_code", "region_std"], how="left")
df = df.merge(base, on="region_std", how="left")

df["price_base"] = df["price_nom"] * (df["I_smr_base"] / df["I_smr_curr"])
df["nmck_base"] = df["nmck_nom"] * (df["I_smr_base"] / df["I_smr_curr"])

n_deflated = df["price_base"].notna().sum()
n_missing = df["price_base"].isna().sum()
print(f"Дефлятировано: {n_deflated} публикаций")

Дефлятировано: 110 публикаций


In [228]:
print(f"Без индекса: {n_missing}")
print(df[df["price_base"].isna()][["registry_number", "quarter_code", "region_std", "publication_name"]].to_string())

Без индекса: 1
       registry_number quarter_code                                   region_std                                                                                                                                                                                                                              publication_name
31  162300005324002661       2024Q4  Нижегородская область; Свердловская область  Строительство объекта «Здание школы для МАОУК «Гимназия «Арт-Этюд» на территории в границах: ул. Шефская - пр. Космонавтов - отвод железной дороги - Калиновский лесопарк (ЖК «Изумрудный бор») в Орджоникидзевском районе г. Екатеринбурга»


In [229]:
print(f"Средний коэффициент дефлятирования: {(df['I_smr_base'] / df['I_smr_curr']).mean():.3f}")

Средний коэффициент дефлятирования: 1.213


### Чувствительность интегрального индекса СМР к весам $(w_L, w_M, w_E)$

Пересчитываю $I^{SMR}_{t,r}$ при нескольких наборах весов (те же исходные квартальные индексы по труду, материалам и машинам) и сравниваю с уже посчитанным базовым `price_base` / `nmck_base` на уровне публикаций: ранговая корреляция Спирмена и относительное отклонение. Объекты linkage не пересобираются — только проверка дефлятирования при смене весов


In [ ]:
idx_path = ROOT / "data_processed" / "minstroy_school_indices_2022Q1_2025Q4_long_filled.csv"
idx_raw = pd.read_csv(idx_path)


def ismr_table_from_weights(w_l: float, w_m: float, w_e: float) -> pd.DataFrame:
    wmap = {"labor": w_l, "materials": w_m, "machines": w_e}
    t = idx_raw.copy()
    t["w"] = t["index_value"] * t["element"].map(wmap)
    return t.groupby(["quarter", "region"], as_index=False)["w"].sum().rename(columns={"w": "I_smr"})


def deflate_with_idx_wide(df_pub, idx_wide):
    base = idx_wide[idx_wide["quarter"] == BASE_QUARTER][["region", "I_smr"]].rename(
        columns={"region": "region_std", "I_smr": "I_smr_base"}
    )
    curr = idx_wide.rename(
        columns={"quarter": "quarter_code", "region": "region_std", "I_smr": "I_smr_curr"}
    )
    m = df_pub.merge(curr, on=["quarter_code", "region_std"], how="left").merge(base, on="region_std", how="left")
    pb = m["price_nom"] * (m["I_smr_base"] / m["I_smr_curr"])
    nb = m["nmck_nom"] * (m["I_smr_base"] / m["I_smr_curr"])
    return pb, nb

In [252]:
cols_needed = ["registry_number", "price_nom", "nmck_nom", "quarter_code", "region_std"]
df_defl = df[cols_needed].copy()

weight_scenarios = [
    ("базовый (как выше)", W_LABOR, W_MATERIALS, W_MACHINES),
    ("материалы+", 0.20, 0.70, 0.10),
    ("труд+", 0.30, 0.60, 0.10),
    ("1/3 каждый элемент", 1 / 3, 1 / 3, 1 / 3),
]

base_pb, base_nb = df["price_base"], df["nmck_base"]
rows = []

for label, wl, wm, we in weight_scenarios:
    iw = ismr_table_from_weights(wl, wm, we)
    pb_alt, nb_alt = deflate_with_idx_wide(df_defl, iw)

    m_pb = base_pb.notna() & pb_alt.notna()
    m_nb = base_nb.notna() & nb_alt.notna()

    rho_pb = base_pb[m_pb].corr(pb_alt[m_pb], method="spearman")
    rho_nb = base_nb[m_nb].corr(nb_alt[m_nb], method="spearman")

    relpb = (pb_alt[m_pb] / base_pb[m_pb] - 1).abs()
    relnb = (nb_alt[m_nb] / base_nb[m_nb] - 1).abs()

    rows.append(
        {
            "сценарий": label,
            "rho_price_base": rho_pb,
            "max_abs_d_pct_price": 100 * float(relpb.max()) if len(relpb) else float("nan"),
            "p90_abs_d_pct_price": 100 * float(relpb.quantile(0.9)) if len(relpb) else float("nan"),
            "rho_nmck_base": rho_nb,
            "max_abs_d_pct_nmck": 100 * float(relnb.max()) if len(relnb) else float("nan"),
        }
    )

sens_w = pd.DataFrame(rows)
fmt = {
    "rho_price_base": lambda x: f"{x:.6f}",
    "rho_nmck_base": lambda x: f"{x:.6f}",
    "max_abs_d_pct_price": lambda x: f"{x:.2f}",
    "p90_abs_d_pct_price": lambda x: f"{x:.2f}",
    "max_abs_d_pct_nmck": lambda x: f"{x:.2f}",
}
print(sens_w.to_string(index=False, formatters=fmt))

          сценарий rho_price_base max_abs_d_pct_price p90_abs_d_pct_price rho_nmck_base max_abs_d_pct_nmck
базовый (как выше)       1.000000                0.00                0.00      1.000000               0.00
        материалы+       0.999531                2.66                1.89      0.999585               2.66
             труд+       0.999883                2.26                1.59      0.999847               2.26
1/3 каждый элемент       0.999197                7.59                2.91      0.999215               7.59


Спирмен между базовым и любым из 3 альтернативных сценариев > 0,999 и по price_base, и по nmck_base. Порядок публикаций после дефляции почти не зависит от того, взяли ли «материалы+», «труд+» или равные веса

Смена весов - это в основном масштабный сдвиг дефлированных сумм, а не перестроение ранжирования (максимальнаям дельта 2,7% и 2,3%, у 90% наблюдений при (0,20; 0,70; 0,10) и (0,30; 0,60; 0,10)). Сценарий (1/3; 1/3; 1/3) — наихудший по уровню

Для бенчмаркинга, выбросов и знаков корреляций базовые веса выглядят нормальными: ранги стабильны. Для сравнения двух соседних объектов по абсолютной удельной стоимости при спорном выборе весов нужно помнить про несколько процентов сдвига (и до ~8% в равновесовом варианте в худших точках)

Сохраняю датасет на уровне публикаций

In [230]:
cls_cols = cls[["registry_number", "work_type", "llm_reason"]].copy()
cls_cols["registry_number"] = cls_cols["registry_number"].astype(str)
df = df.merge(cls_cols, on="registry_number", how="left")

PUB_COLS = [
    "registry_number", "purchase_code", "publication_name", "work_type",
    "region_std", "Город поставки", "date_pub", "year", "quarter", "quarter_code",
    "Тип торгов", "Торговая площадка", "nmck_nom", "contract_price_nom", "price_nom", 
    "price_source", "nmck_base", "price_base", "discount_pct", "has_contract_price",
    "I_smr_curr", "I_smr_base", "places", "places_source_file", "area_m2", 
    "address_text", "confidence", "package_purchase", "llm_reason",
]

df_pub = df[PUB_COLS].copy()
OUT_PUB = ROOT / "data_processed" / "analysis_publications.csv"
df_pub.to_csv(OUT_PUB, index=False)

print(f"Строк: {len(df_pub)}, колонок: {len(df_pub.columns)}")

Строк: 111, колонок: 29


In [231]:
print(df_pub.dtypes)

registry_number               object
purchase_code                 object
publication_name              object
work_type                     object
region_std                    object
Город поставки                object
date_pub              datetime64[ns]
year                           int32
quarter                        int32
quarter_code                  object
Тип торгов                    object
Торговая площадка             object
nmck_nom                     float64
contract_price_nom           float64
price_nom                    float64
price_source                  object
nmck_base                    float64
price_base                   float64
discount_pct                 float64
has_contract_price              bool
I_smr_curr                   float64
I_smr_base                   float64
places                       float64
places_source_file            object
area_m2                      float64
address_text                  object
confidence                    object
p

### Объединяю публикации в объекты строительства

Один и тот же объект мог закупаться несколькими публикациями (например, отдельно ПИР и СМР). Чтобы получить полную стоимость объекта, нужно их объединить. Объединяю по 3 признакам: регион + город + число мест, если при этом названия похожи

In [232]:
SIMILARITY_THRESHOLD = 0.65

NOISE = ["выполнение", "работ", "строительству", "объекта",
         "капитального", "строительства", "капитальное",
         "мбоу", "маоу", "моу", "гбу"]

def sim(a, b):
    def clean(s):
        s = str(s).lower()
        for w in NOISE:
            s = s.replace(w, " ")
        return re.sub(r"\s+", " ", s).strip()
    return SequenceMatcher(None, clean(a), clean(b)).ratio()

df_link = df_pub[df_pub["places"].notna()].copy()
df_noplace = df_pub[df_pub["places"].isna()].copy()

print(f"Публикаций с places для linkage: {len(df_link)}")
print(f"Публикаций без places (отдельная группа): {len(df_noplace)}")

Публикаций с places для linkage: 109
Публикаций без places (отдельная группа): 2


In [233]:
records = df_link.reset_index(drop=True)
n = len(records)

parent = list(range(n))

def find(x):
    while parent[x] != x:
        parent[x] = parent[parent[x]]
        x = parent[x]
    return x

def union(x, y):
    px, py = find(x), find(y)
    if px != py:
        parent[py] = px

groups = records.groupby(["region_std", "Город поставки", "places"]).groups

In [234]:
merge_count = 0
for (region, city, places), idxs in groups.items():
    idxs = list(idxs)
    if len(idxs) == 1:
        continue
    # проверяем, похожи ли названия - если да, считаем их одним объектом
    for i in range(len(idxs)):
        for j in range(i + 1, len(idxs)):
            ni = records.loc[idxs[i], "publication_name"]
            nj = records.loc[idxs[j], "publication_name"]
            score = sim(ni, nj)
            if score >= SIMILARITY_THRESHOLD:
                union(idxs[i], idxs[j])
                merge_count += 1

records["object_id"] = [f"obj_{find(i):04d}" for i in range(n)]

df_noplace = df_noplace.copy()
df_noplace["object_id"] = [f"obj_noplace_{i:04d}" for i in range(len(df_noplace))]

df_pub2 = pd.concat([records, df_noplace], ignore_index=True)

n_objects = df_pub2["object_id"].nunique()
n_pubs = len(df_pub2)
print(f"Публикаций: {n_pubs}")
print(f"Объектов: {n_objects}")
print(f"Слияний выполнено: {merge_count}")

Публикаций: 111
Объектов: 72
Слияний выполнено: 61


In [235]:
obj_sizes = df_pub2.groupby("object_id").size()
multi = obj_sizes[obj_sizes > 1].index
print(f"Объектов из >1 публикации: {len(multi)}")

Объектов из >1 публикации: 23


### Агрегирую данные на уровень объекта и считаю удельную стоимость

In [236]:
def agg_object(grp):
    # берем только публикации с реальным контрактом (есть цена победителя)
    contracts = grp[grp["has_contract_price"]]

    if len(contracts) > 0:
        # суммируем контракты (могут быть разные этапы одного объекта)
        cost_nom = contracts["contract_price_nom"].sum()
        cost_base = contracts["price_base"].sum() if contracts["price_base"].notna().any() else None
        price_src = "contract"
    else:
        # ни одна публикация не завершилась контрактом — берем НМЦК из одной (первой по дате)
        first = grp.sort_values("date_pub").iloc[0]
        cost_nom = first["nmck_nom"]
        cost_base = first["nmck_base"] if pd.notna(first["nmck_base"]) else None
        price_src = "nmck"

    return pd.Series({
        "n_publications": len(grp),
        "registry_numbers": ";".join(grp["registry_number"].astype(str)),
        "work_type": grp["work_type"].mode().iloc[0] if len(grp) > 0 else None,
        "region_std": grp["region_std"].iloc[0],
        "city": grp["Город поставки"].iloc[0],
        "year": grp["year"].min(),
        "quarter_code": grp.sort_values("date_pub")["quarter_code"].iloc[0],
        "places": grp["places"].dropna().iloc[0] if grp["places"].notna().any() else None,
        "area_m2": grp["area_m2"].dropna().iloc[0] if grp["area_m2"].notna().any() else None,
        "cost_nom": cost_nom,
        "cost_base": cost_base,
        "nmck_nom": grp.sort_values("date_pub")["nmck_nom"].iloc[0],
        "nmck_base": grp.sort_values("date_pub")["nmck_base"].iloc[0],
        "any_package": grp["package_purchase"].any(),
        "price_source": price_src,
        "n_contracts": len(contracts),
        "discount_pct_mean": grp["discount_pct"].mean(),
        "discount_pct_max": grp["discount_pct"].max(),
        "address_text": grp["address_text"].dropna().iloc[0] if grp["address_text"].notna().any() else None,
        "publication_name": grp.sort_values("date_pub")["publication_name"].iloc[0],
    })

df_obj = df_pub2.groupby("object_id").apply(agg_object, include_groups=False).reset_index()

df_obj["unit_cost_base"] = df_obj["cost_base"] / df_obj["places"]
df_obj["unit_cost_nom"] = df_obj["cost_nom"] / df_obj["places"]
df_obj["unit_nmck_base"] = df_obj["nmck_base"] / df_obj["places"]
df_obj["unit_cost_m2"] = df_obj["cost_base"] / df_obj["area_m2"]

print(f"Объектов итого: {len(df_obj)}")
print(f"С удельной стоимостью (places + cost_base): {df_obj['unit_cost_base'].notna().sum()}")
print(f"  из них на основе контракта: {(df_obj['price_source'] == 'contract').sum()}")
print(f"  из них на основе НМЦК: {(df_obj['price_source'] == 'nmck').sum()}")

Объектов итого: 72
С удельной стоимостью (places + cost_base): 69
  из них на основе контракта: 39
  из них на основе НМЦК: 33


Объектов с несколькими контрактами (суммируются):

In [237]:
multi_contract = df_obj[
    (df_obj["n_publications"] > 1) & (df_obj["n_contracts"] > 1)
].sort_values("n_contracts", ascending=False)

for _, obj in multi_contract.iterrows():
    pubs = df_pub2[df_pub2["object_id"] == obj["object_id"]]
    contracts = pubs[pubs["has_contract_price"]]
    oid = obj["object_id"]
    places = obj["places"]
    total = obj["cost_base"] / 1e6
    n = len(contracts)
    
    print(f"{oid} | places={places:.0f} | итого={total:.0f}M (сумма {n} контрактов)")
    for _, r in contracts.iterrows():
        cp = r["contract_price_nom"] / 1e6
        name = str(r["publication_name"])[:60]
        print(f"  + {cp:.0f}M | {name}")
    print()

obj_0034 | places=14 | итого=2246M (сумма 5 контрактов)
  + 369M | Выполнение строительно-монтажных и прочих работ, в том числе
  + 122M | Выполнение строительно-монтажных и прочих работ, в том числе
  + 722M | Выполнение строительно-монтажных и прочих работ, в том числе
  + 722M | Выполнение строительно-монтажных и прочих работ, в том числе
  + 61M | Выполнение строительно-монтажных и прочих работ, в том числе

obj_0036 | places=2025 | итого=1076M (сумма 4 контрактов)
  + 391M | Выполнение строительно-монтажных и прочих работ, в том числе
  + 126M | Выполнение строительно-монтажных и прочих работ, в том числе
  + 256M | Выполнение строительно-монтажных и прочих работ, в том числе
  + 195M | Выполнение строительно-монтажных и прочих работ, в том числе

obj_0035 | places=1224 | итого=3571M (сумма 3 контрактов)
  + 200M | Выполнение строительно-монтажных и прочих работ, в том числе
  + 1598M | Выполнение строительно-монтажных и прочих работ, в том числе
  + 1652M | Выполнение строительно

Объекты без контракта (взята НМЦК одной публикации):

In [238]:
nmck_only = df_obj[df_obj["price_source"] == "nmck"].sort_values("cost_nom", ascending=False)
for _, obj in nmck_only.iterrows():
    pubs = df_pub2[df_pub2["object_id"] == obj["object_id"]]
    oid = obj["object_id"]
    places = obj["places"]
    nmck = obj["cost_nom"] / 1e6
    n = len(pubs)
    
    print(f"{oid} | places={places:.0f} | НМЦК={nmck:.0f}M ({n} публикация, победителя нет)")
    for _, r in pubs.iterrows():
        rn = r["registry_number"]
        nm = r["nmck_nom"] / 1e6
        flag = "контракт" if r["has_contract_price"] else "только НМЦК"
        name = str(r["publication_name"])[:55]
        print(f"  {rn} | {nm:.0f}M | {flag} | {name}")
    print()


obj_0066 | places=1440 | НМЦК=2917M (1 публикация, победителя нет)
  356500001425003735 | 2917M | только НМЦК | ВЫПОЛНЕНИЕ РАБОТ ПО ОБЪЕКТУ КАПИТАЛЬНОГО СТРОИТЕЛЬСТВА:

obj_0026 | places=1755 | НМЦК=2130M (1 публикация, победителя нет)
  153300066923000085 | 2130M | только НМЦК | Выполнение проектно-изыскательских и строительно-монтаж

obj_0097 | places=1400 | НМЦК=1936M (1 публикация, победителя нет)
  815500000525002535 | 1936M | только НМЦК | Строительство объекта "Средняя общеобразовательная школ

obj_0090 | places=1100 | НМЦК=1761M (1 публикация, победителя нет)
  813500000125009132 | 1761M | только НМЦК | зз-0017-23424 - 2025 Выполнение работ по строительству 

obj_0065 | places=1100 | НМЦК=1649M (1 публикация, победителя нет)
  356500001425003241 | 1649M | только НМЦК | ВЫПОЛНЕНИЕ РАБОТ ПО ОБЪЕКТУ КАПИТАЛЬНОГО СТРОИТЕЛЬСТВА:

obj_0062 | places=825 | НМЦК=1502M (2 публикация, победителя нет)
  356500001424007477 | 1502M | только НМЦК | Строительство школы в г.Нытва
  356500001424

Несколько публикаций, контракт только у одной — берем ее цену:

In [239]:
one_contract = df_obj[
    (df_obj["n_publications"] > 1) & (df_obj["n_contracts"] == 1)
].sort_values("n_publications", ascending=False)

for _, obj in one_contract.iterrows():
    pubs = df_pub2[df_pub2["object_id"] == obj["object_id"]]
    oid = obj["object_id"]
    places = obj["places"]
    cost = obj["cost_base"] / 1e6

    print(f"{oid} | places={places:.0f} | взята цена одного контракта = {cost:.0f}M")
    for _, r in pubs.iterrows():
        flag = "→ КОНТРАКТ (берется)" if r["has_contract_price"] else "игнорируется"
        nm = r["nmck_nom"] / 1e6
        cp = f"{r['contract_price_nom']/1e6:.0f}M" if r["has_contract_price"] else f"нмцк {nm:.0f}M"
        name = str(r["publication_name"])[:60]
        print(f"  {flag}: {cp} | {name}")
    print()

obj_0052 | places=1100 | взята цена одного контракта = 2290M
  игнорируется: нмцк 2319M | Выполнение работ по строительству объекта: "Школа на 1100 ме
  → КОНТРАКТ (берется): 2075M | Выполнение работ по строительству объекта: "Школа на 1100 ме
  игнорируется: нмцк 1323M | Выполнение работ по строительству объекта: "Школа на 1100 ме
  игнорируется: нмцк 1323M | Выполнение работ по строительству объекта: "Школа на 1100 ме

obj_0001 | places=640 | взята цена одного контракта = 996M
  игнорируется: нмцк 677M | Выполнение строительно-монтажных, пусконаладочных работ, пос
  игнорируется: нмцк 746M | Выполнение строительно-монтажных, пусконаладочных работ, пос
  → КОНТРАКТ (берется): 727M | Выполнение строительно-монтажных, пусконаладочных работ, пос

obj_0095 | places=300 | взята цена одного контракта = 666M
  игнорируется: нмцк 612M | Строительство здания средней общеобразовательной школы на 30
  игнорируется: нмцк 612M | Строительство здания средней общеобразовательной школы на 30
  → КОНТ

Описательная статистика unit_cost_base (тыс. руб./место)

In [240]:
print((df_obj["unit_cost_base"] / 1000).describe().round(0))

count        69.0
mean       4154.0
std       19149.0
min           7.0
25%        1039.0
50%        1570.0
75%        2292.0
max      160428.0
Name: unit_cost_base, dtype: float64


In [241]:
OUT_OBJ = ROOT / "data_processed" / "analysis_objects.csv"
df_obj.to_csv(OUT_OBJ, index=False)
df_pub2.to_csv(ROOT / "data_processed" / "analysis_publications.csv", index=False)
print(f"Строк: {len(df_obj)}, колонок: {len(df_obj.columns)}")

Строк: 72, колонок: 25


## Итого:

Изначально 157 публикаций, после LLM фильтрации - 111

In [242]:
print(f"Объектов строительства: {len(df_obj)}")
print(f"  из них с unit_cost_base: {df_obj['unit_cost_base'].notna().sum()}")
print(f"  из них без places: {df_obj['places'].isna().sum()}")

Объектов строительства: 72
  из них с unit_cost_base: 69
  из них без places: 2


In [243]:
print("Регионы в выборке:")
print(df_obj.groupby("region_std").size().sort_values(ascending=False))

Регионы в выборке:
region_std
Пермский край                                  11
Республика Башкортостан                        11
Нижегородская область                           7
Республика Мордовия                             7
Республика Татарстан (Татарстан)                7
Оренбургская область                            6
Удмуртская Республика                           5
Саратовская область                             4
Чувашская Республика - Чувашия                  4
Республика Марий Эл                             3
Кировская область                               2
Пензенская область                              2
Ульяновская область                             2
Нижегородская область; Свердловская область     1
dtype: int64


In [244]:
print("Годы:")
print(df_obj.groupby("year").size())

Годы:
year
2022    22
2023    13
2024    16
2025    21
dtype: int64


### Финальная выборка для основного анализа

Объекты с несколькими контрактами (n_contracts > 1) исключаются из основной выборки:
при суммировании нескольких контрактов нельзя быть уверенным, что не будет ложных слияний (один объект мог ошибочно получить публикации от разных школ с похожими названиями и одинаковой мощностью). Объекты с n_contracts == 1 при нескольких публикациях — это перезапуски одного тендера, у них есть ровно одна цена контракта на полный объем работ, риска удвоения нет

Финальная выборка: n_contracts <= 1 (или price_source == "nmck")

Исключаем объекты, где суммируются несколько контрактов:

In [245]:
df_obj_core = df_obj[df_obj["n_contracts"] <= 1].copy()

print(f"Полная выборка объектов: {len(df_obj)}")
print(f"  из них с несколькими контрактами (>1): {(df_obj['n_contracts'] > 1).sum()} - исключаются")

Полная выборка объектов: 72
  из них с несколькими контрактами (>1): 11 - исключаются


In [254]:
print(f"Финальная выборка: {len(df_obj_core)}")
print(f"  с ценой контракта: {(df_obj_core['price_source'] == 'contract').sum()}")
print(f"  по НМЦК: {(df_obj_core['price_source'] == 'nmck').sum()}")
print(f"  с unit_cost_base: {df_obj_core['unit_cost_base'].notna().sum()}")
print(f"  с n_publications > 1 (несколько публикаций на объект): {(df_obj_core['n_publications'] > 1).sum()}")

Финальная выборка: 61
  с ценой контракта: 28
  по НМЦК: 33
  с unit_cost_base: 58
  с n_publications > 1 (несколько публикаций на объект): 12


In [247]:
print("Описательная статистика unit_cost_base (тыс.руб./место) — финальная выборка:")
print((df_obj_core["unit_cost_base"] / 1000).describe().round(0))

Описательная статистика unit_cost_base (тыс.руб./место) — финальная выборка:
count      58.0
mean     1742.0
std      1350.0
min         7.0
25%      1085.0
50%      1559.0
75%      2061.0
max      8092.0
Name: unit_cost_base, dtype: float64


In [248]:
OUT_CORE = ROOT / "data_processed" / "analysis_objects_core.csv"
df_obj_core.to_csv(OUT_CORE, index=False)